In [ ]:
import pandas as pd
# import sweetviz as sv
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import train_test_split
import seaborn as sns
#from pandas_profiling import ProfileReport
import utils

In [ ]:
from IPython.display import display, HTML
display(HTML("<style>.container { width:90% !important; }</style>"))

# 1. Description
The data we have is from the WESAD study and contains labled data (stress/no stress). The data contains data from differente sensors: ACC (accelerometer), BVP (blood volume pulse), EDA (electrodermal activity), TEMP (temperature). In the following, we look at the features created with the FLIRT (https://flirt.readthedocs.io/en/latest/) library (see other script).

Note: there might be other potential features to be calculated on the raw data, for example via tsfresh (https://tsfresh.readthedocs.io/en/latest/index.html) or TSFEL (https://tsfel.readthedocs.io/en/latest/). However, FLIRT was specifically developed with the wrist sensor used in the two dataset used here, so we can reasonably expect it to produce meaningful features based on the available data.

# 2. Data Source

In [ ]:
# load data - features calculated with Flirt with
# window_length = 60 and
# window_step_size = 10
df = pd.read_parquet('data-input/flirt-wesad-acc-bvp-eda-temp60-10.parquet')

In [ ]:
df.shape

In [ ]:

df.head(3)

In [ ]:
# there are no missing values in the dataset
df.isnull().sum().value_counts()

In [ ]:
df['subject'].value_counts()

In [ ]:
df['label'].value_counts(normalize=True)

# 3. Train-test split
We perform the train-test split before we conduct EDA on the train set. Thus, we avoid data leakage from the test set.

In [ ]:
df_train, df_test = utils.create_train_test(df, 5, 'subject', 'label')

In [ ]:
df_train.shape

In [ ]:
df_test.shape

In [ ]:
df_train['label'].value_counts(normalize=True)

In [ ]:
df_test['label'].value_counts(normalize=True)

# 4. EDA
## 4.1 Looking into data

In [ ]:
# we do not have categorical features, only int (count) and float
df_train.dtypes.value_counts()

In [ ]:
df_train.describe()

In [ ]:
# remove rows with only one value for each row
#overall_length = len(df_train)
columns = df_train.columns.tolist()
constant_columns = []

for c in columns:
    unique_in_column = len(df_train[c].unique())
    
    #if unique_in_column/overall_length < 0.1 and c != 'label' and c != 'subject':
    if unique_in_column == 1:
        constant_columns.append(c)

In [ ]:
constant_columns

In [ ]:
# remove rows where we cannot calculate sandard deviation of the column

df_desc = df_train.describe()
columns_describe = df_train.describe().columns.tolist()
no_std_columns = []

for c in columns:
    std = df_desc[c]['std']

    #if unique_in_column/overall_length < 0.1 and c != 'label' and c != 'subject':
    if np.isnan(std):
        no_std_columns.append(c)

In [ ]:
no_std_columns

In [ ]:
columns_to_drop = list((set(constant_columns).union(set(no_std_columns))))

In [ ]:
columns_to_drop

In [ ]:
df_train = df_train.drop(columns=columns_to_drop)

## 4.2 Correlations

In [ ]:
plt.figure(figsize=(35, 35))
corr = df_train.corr(method='spearman')
heatmap = sns.heatmap(corr.sort_values(by='label', ascending=False),
                      vmin=-1, vmax=1, annot=True, fmt='.1g', cmap='BrBG')
heatmap.set_title('Features correlating with stress label', fontdict={'fontsize':15}, pad=16);

# 5. Documenting data lineage
The dataset contains

BVP (blood volume pulse) sensor data in 64hz
ACC: accelerometer data (x, y, z values) in 32hz
EDA (electrodermal activity) in 4hz
TEMP (temperature) in 4hz
The dataset is labeled (stress/no stress).

Script 01-extract-data-from-wesad-dataset downloads the wesad study dataset. It unzips all included files. The raw data is stored in a pickle file provided from the researchers publishing the dataset. We load it from there, as well as the labels (stress/no stress). The results of the extraction are stored as a parquet file.

In script 02-calculate-features, we calculate features with the FLIRT library (https://flirt.readthedocs.io/en/latest/). In this notebook you're currently reading, we perform EDA. In the following steps, we might want to go back to feature calculation and calculate other/more features.

# 6. Observations from EDA
Looking into the dataset
Given the windows size and step size, we have 2765 rows and 264 features.

We could also calculate feature via tsfresh and/or TSFEL; and we could try different parameters for window_length and window_step_size when using FLIRT.

There are no missing values.

We have 64% of negative cases (no stress) and 36% of positive cases (stress) in our data - we have to account for this when building and evaluating the model, e.g., by using appropriate evaluation metrics for imbalanced data.

We do not have categorical variables, only numerical (count and float).

There are a some columns that we drop, because they either have have the same value for each row, or it is impossible to calculate the standard deviation on them:

eda_EDA_n_sign_changes
temp_TEMP_peaks
acc_y_entropy
acc_l2_n_sign_changes
acc_x_entropy
acc_z_entropy
temp_l2_n_sign_changes
bvp_BVP_entropy
temp_TEMP_n_sign_changes
temp_l2_peaks
eda_l2_n_sign_changes
The ranges of the values are quite far from each other - we should normalize/standardize.

## Correlations
There are several correlated features. Because of the amount of features, we should apply an automated method for deciding which features to keep.